# Track 2 starter: Data-quality flags on synthetic results
Rule-based checks + an unsupervised model. Outputs are **flags for human review**, never declarations of fraud or results.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, numpy as np
from ai4elections import load_csv, load_jsonl
pd.set_option('display.max_columns', 30)
from ai4elections.checks import integrity_flags

In [ ]:
res = load_csv('results_synthetic.csv')
flagged = integrity_flags(res)
flag_cols = [c for c in flagged.columns if c.startswith('flag_')]
flagged[flag_cols].sum()

In [ ]:
# Evaluate against held-out labels (available only for practice; real data has no answer key)
lab = load_csv('anomaly_labels_HELD_OUT.csv')
m = flagged.merge(lab, on='pu_id')
m['any_flag'] = m[flag_cols].any(axis=1)
m['is_anom'] = m['anomaly_type'] != 'none'
print(pd.crosstab(m['is_anom'], m['any_flag']))
print(m[m.is_anom].groupby('anomaly_type')['any_flag'].mean())

In [ ]:
# Unsupervised: Isolation Forest on turnout & vote shares
from sklearn.ensemble import IsolationForest
P = ['votes_A','votes_B','votes_C','votes_D','votes_E']
X = pd.DataFrame({'turnout': res.accredited_voters/res.registered_voters})
valid = res[P].sum(axis=1).replace(0,np.nan)
for p in P: X['share_'+p[-1]] = res[p]/valid
X = X.fillna(0)
iso = IsolationForest(contamination=0.05, random_state=0).fit(X)
m['iso_flag'] = iso.predict(X) == -1
print(pd.crosstab(m['is_anom'], m['iso_flag']))

## Your turn
- Which anomaly types do rules catch vs the model? Combine them.
- Report precision/recall AND the cost of false alarms on reviewers' time.
- Never train or test on live or unauthorised data. See `docs/01_technical_briefs/track2_electoral_data_intelligence.md`.